In [1]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import datetime
import keras_tuner as kt

In [2]:
(X_train_full,y_train_full),(X_test,y_test) = keras.datasets.mnist.load_data()
X_train_full.shape,X_test.shape,y_train_full.shape,y_test.shape

((60000, 28, 28), (10000, 28, 28), (60000,), (10000,))

In [3]:
X_train,X_valid = X_train_full[:50000,:], X_train_full[50000:,:]
y_train,y_valid = y_train_full[:50000], y_train_full[50000:] 

X_train , X_valid,X_test = X_train/255.0 , X_valid/255.0,X_test/255.0
X_train.shape,X_valid.shape,X_test.shape,y_train.shape,y_valid.shape,y_test.shape

((50000, 28, 28),
 (10000, 28, 28),
 (10000, 28, 28),
 (50000,),
 (10000,),
 (10000,))

In [10]:
y_train[0]

np.uint8(5)

<span style="color:red;font-weight:bold;font-size:20px">
下边这个code是一个固定的模型，隐藏层是自己修改代码决定的，这种不太好找最好的参数，即最好的隐藏层数，以及每层的神经元数量。
但内容还是比较全面的，包括了模型的构建，complied，callback里边的early stopping、checkpoint和可视化，在下一个code就是可视化的结果。
</span>

In [11]:
model = keras.models.Sequential([
    keras.layers.Input(shape = (28,28)),
    keras.layers.Flatten(),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10,activation = "softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)
#加入可视化

log_dir = os.path.join("logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))  #时间机器

tensorboard_cb = keras.callbacks.TensorBoard(
    log_dir=log_dir, 
    histogram_freq=1  # 每一轮都统计一下权重的直方图（可选，想看详细的就设为1）
)

callbacks=[
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True),
    tensorboard_cb
]

history = model.fit(X_train,y_train,epochs=100,
                    validation_data=(X_valid,y_valid),
                   callbacks=callbacks,)


Epoch 1/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 719us/step - accuracy: 0.9211 - loss: 0.2651 - val_accuracy: 0.9586 - val_loss: 0.1436
Epoch 2/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 644us/step - accuracy: 0.9645 - loss: 0.1163 - val_accuracy: 0.9663 - val_loss: 0.1133
Epoch 3/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 639us/step - accuracy: 0.9743 - loss: 0.0831 - val_accuracy: 0.9673 - val_loss: 0.1029
Epoch 4/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 654us/step - accuracy: 0.9797 - loss: 0.0637 - val_accuracy: 0.9769 - val_loss: 0.0794
Epoch 5/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 645us/step - accuracy: 0.9826 - loss: 0.0536 - val_accuracy: 0.9739 - val_loss: 0.0942
Epoch 6/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 674us/step - accuracy: 0.9866 - loss: 0.0423 - val_accuracy: 0.9702 - val_loss: 0.1056
Epoch 7/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 657us/step - accuracy: 0.9874 - loss: 0.0383 - val_accuracy: 0.9744 - val_loss: 0.1004
Epoch 8/100
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1s 650us/step - accuracy: 

In [23]:
#可视化结果
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<span style="color:red">下边两个code，一个是看后台有无tensorboard在运行，一个是关闭，关闭的话，后边的数字改成 pip后边的数字即可</span>

In [26]:
from tensorboard import notebook

notebook.list()

No known TensorBoard instances running.


In [25]:
! kill 3633

In [15]:
y_pre = model.predict(X_test)
loss, accuracy = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 260us/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 301us/step - accuracy: 0.9751 - loss: 0.0794


In [17]:
y_pre[1]

array([8.9614455e-11, 2.5380741e-05, 9.9997443e-01, 1.9673185e-07,
       1.4285921e-10, 1.1360373e-10, 1.3279171e-09, 2.3226827e-08,
       4.4004094e-09, 5.3787872e-13], dtype=float32)

对于这个结果，显然不是我们想要的分类结果，想要的就是最后的能看懂的分类结果

In [18]:
y_pre_f = np.argmax(y_pre, axis=1) #这一步将结果改成最大的数字是第几位，也就是我们想要的结果。

In [19]:
y_pre_f[:20],y_test[:20]

(array([7, 2, 1, 0, 4, 1, 4, 9, 6, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4]),
 array([7, 2, 1, 0, 4, 1, 4, 9, 5, 9, 0, 6, 9, 0, 1, 5, 9, 7, 3, 4],
       dtype=uint8))

In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 299,132 (1.14 MB)

 Trainable params: 99,710 (389.49 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 199,422 (779.00 KB)

 <span style="color:red;font-size:20px">如何调取训练好的model ，不需要从新训练，直接读取保存的model</span>

```python
import tensorflow as tf
import numpy as np
from tensorflow import keras
```
 1. 【关键】读取模型 (就像读档游戏)
 不需要重新 build_model，也不需要重新 compile，它全记住了

```python
new_model = keras.models.load_model("mnist_best.keras")
```
 3. 准备一张测试图 (这里假设随便拿测试集的一张)
 注意！必须也要除以 255.0，否则模型看不懂

```python
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()

sample_image = X_test[0] / 255.0  
```
 5. 预测
 注意：模型输入必须是 (样本数, 28, 28)，所以要给单张图片加一个维度
```python
sample_batch = np.expand_dims(sample_image, axis=0) # 变成 (1, 28, 28)

predictions = new_model.predict(sample_batch)

predicted_number = np.argmax(predictions)

print(f"模型预测这张图是数字: {predicted_number}")

print(f"真实标签是: {y_test[0]}")
```

# 以下是随机检索最优参数的代码，比较复杂

# keras-tuner 就是这个

```python
def build_model(hp):
    model = keras.models.Sequential()
    # 输入层和 Flatten (这两个通常不用动)
    model.add(keras.layers.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    
    # 核心修改】隐藏层：让 Tuner 决定层数和神经元数量
    # hp.Int('num_layers', 1, 3) 意思是：尝试 1层, 2层, 或 3层
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(keras.layers.Dense(
            # hp.Int 意思是：在 32 到 256 之间选一个整数，步长 32 (即 32, 64, 96...)
            units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
            activation='relu'
        ))
    
    # 4. 输出层 也不需要动
    model.add(keras.layers.Dense(10, activation='softmax'))
    
    # 5. 【核心】编译：让 Tuner 决定学习率
    # log 采样意思是：在 0.0001 到 0.01 之间按数量级挑 (0.01, 0.001, 0.0001)， sampling='log'这个需要加上
    hp_learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    
    model.compile(
        loss="sparse_categorical_crossentropy",
        # 注意：这里不能写字符串 "adam"，因为我们要往里面传参数，必须实例化类
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        metrics=["accuracy"]
    )
    
    return model
```

<span style="color:red">这里的搜索器参数较多，这个随机搜索和网格检索，区别就是，网格检索是要把所有的都跑一遍才出结果，这个好像是不需要跑完全部的参数</span>

1. 创建搜索器
```python
tuner = kt.RandomSearch(
    build_model,                     # 把刚才写的函数传进去
    objective='val_accuracy',        # 我们的目标是验证集准确率最大化
    max_trials=10,                   # 总共试 10 种不同的模型组合
    executions_per_trial=1,          # 每种组合跑 1 次 (如果怕偶然性可以设为2)
    directory='my_tuning_dir',       # 搜索日志存哪
    project_name='mnist_tuning'      # 项目名字
)
```
2. 准备 Callbacks (和你之前的一样，但 TensorBoard 路径 Tuner 会自动管理，可以简化)
在搜索阶段，通常只需要 EarlyStopping，ModelCheckpoint 先不用（最后确定了再存）
```python
search_callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.TensorBoard(log_dir="tmp/tb_logs") # 搜索时的临时日志
]
```
3. 开始搜索！ (这就代替了原来的 model.fit),
这里的参数和 model.fit 几乎一样
```python
tuner.search(
    X_train, y_train,
    epochs=20,                       # 搜索时每种模型最多跑 20 轮
    validation_data=(X_valid, y_valid),
    callbacks=search_callbacks
)
```

 --- 补充代码：查看搜索结果 ---

看看最好的参数是啥
```python
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
搜索完成！
最佳隐藏层数量: {best_hps.get('num_layers')}
最佳学习率: {best_hps.get('learning_rate')}
""")
```
打印每一层的最佳神经元数量
```python
for i in range(best_hps.get('num_layers')):
    print(f"第 {i} 层最佳神经元数量: {best_hps.get(f'units_{i}')}")
```
直接获取最好的模型（不需要重新训练）
```python
best_model = tuner.get_best_models(num_models=1)[0]
```
看看它的准确率
```python
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"最佳模型在测试集上的准确率: {accuracy}")
```